# Ocean Wind Field API — Sentinel-1 SAR

Retrieves the 5 km ocean wind field (direction and speed) over a coastal area for a chosen period.

**How to use** — run the cells from top to bottom (▶ button of each cell):
1. **Setup** (once per session): downloads the code and the trained model, then asks you to sign in to
   Google Earth Engine and Copernicus Marine.
2. **Step 1:** draw the area of interest on the map (maximum 50 km × 50 km, mostly sea).
3. **Step 2:** enter the start and end date and run the cell; the wind-field maps appear below it.
4. **Step 3:** download the results as a zip file.

**Accounts (free):** a Google account registered for Earth Engine with a Cloud project
([sign up](https://earthengine.google.com/)), and a Copernicus Marine account ([sign up](https://marine.copernicus.eu/)).

**Outputs (per Sentinel-1 acquisition)**

| File | Content |
|---|---|
| `wind_vectors.geojson` | one point per 5 km sea patch: wind direction (from, °), speed (m/s), u, v, quality flag |
| `wind_field_*.csv` | the same vectors with all intermediate values |
| `map_*.png` | wind-vector map, land masked |
| `summary_*.json` | acquisition details and comparison with Copernicus Marine wind |

**Method:** Sentinel-1 IW GRDH VV backscatter is read from Google Earth Engine, averaged to 100 m and split into
5 km patches. The wind direction is retrieved by a ResNet ensemble, the 180° ambiguity is resolved with Copernicus
Marine wind, and the wind speed is obtained by inverting the CMOD5.N geophysical model function.

**Notes**
- Sentinel-1 revisits an area every few days; a period of at least two weeks is recommended (maximum 61 days).
- Black arrows are *good* quality; white arrows are *low* quality (weak wind-streak texture, typically wind below ~5 m/s).
- Clear all outputs before sharing the notebook, as the sign-in outputs show account names.

In [ ]:
#@title Setup (run once per session)
#@markdown Enter your Google Earth Engine Cloud project id, then run this cell.
EE_PROJECT = ""   #@param {type:"string"}

import os, sys

REPO_URL = "https://github.com/aditonu/Resnet_Wind_Products.git"
REPO_DIR = "/content/Resnet_Wind_Products"
if os.path.isdir(REPO_DIR):
    !git -C {REPO_DIR} pull -q
else:
    !git clone -q --depth 1 {REPO_URL} {REPO_DIR}
!pip install -q -r {REPO_DIR}/requirements.txt
os.chdir(f"{REPO_DIR}/src")
sys.path.insert(0, f"{REPO_DIR}/src")

import api_wind, wind_app, copernicusmarine
if not EE_PROJECT.strip():
    raise ValueError("Enter your Earth Engine Cloud project id in EE_PROJECT and run this cell again.")
api_wind.init(ee_project=EE_PROJECT.strip())     # Earth Engine sign-in
copernicusmarine.login()                          # Copernicus Marine sign-in
wind_app.configure(model_dir=f"{REPO_DIR}/models/final_model", out_root="/content/wind_results")
print("Setup complete.")

In [ ]:
#@title Step 1 — Draw the area of interest
#@markdown Run this cell, then draw a rectangle over the sea with the rectangle tool (&#9723;).
#@markdown The box stops growing at 50 km × 50 km. To change the area, draw a new box.
MAP_CENTER_LAT = 12.0   #@param {type:"number"}
MAP_CENTER_LON = 80.4   #@param {type:"number"}
wind_app.show(center=(MAP_CENTER_LAT, MAP_CENTER_LON), zoom=8)

In [ ]:
#@title Step 2 — Choose the dates and get the wind field
#@markdown Dates are in UTC and both are included. Processing takes a few minutes per Sentinel-1 acquisition.
START_DATE = "2024-09-20"   #@param {type:"date"}
END_DATE   = "2024-09-30"   #@param {type:"date"}
results = wind_app.run(START_DATE, END_DATE)

In [ ]:
#@title Step 3 — Download the results (zip)
from google.colab import files
files.download(wind_app.zip_results(zip_path="/content/wind_results.zip"))

## Optional

**Results table** — summary of the request and the wind vectors of the first acquisition.

In [ ]:
#@title Results table
import json
import pandas as pd
if results:
    display(pd.read_csv("/content/wind_results/last_api_summary.csv"))
    first = next((r for r in results if r.get("geojson")), None)
    if first:
        feats = json.load(open(first["geojson"]))["features"]
        display(pd.DataFrame([{"lon": f["geometry"]["coordinates"][0], "lat": f["geometry"]["coordinates"][1],
                               **f["properties"]} for f in feats]))

**Independent checks**
- *Texture check:* compares the network direction with the FFT orientation of the SAR image and with Copernicus.
- *Scatterometer comparison:* compares with near-simultaneous scatterometer winds (ASCAT, HY-2, OceanSat-3; Copernicus Marine L3).

In [ ]:
#@title Independent checks
for r in results:
    if "run_dir" in r:
        !python check_direction.py --run_dir "{r['run_dir']}"
        !python scat_compare.py --run_dir "{r['run_dir']}"